# Capítulo 4 · Transformada de Fourier Cuántica (QFT)

## Objetivos

1. Comprender la QFT como versión cuántica de la DFT clásica.
2. Construir el circuito de la QFT para $n$ qubits.
3. Verificar la unitariedad y comparar con la matriz analítica.
4. Aplicar la QFT en un ejemplo numérico y visualizar el resultado.

---

## 4.1 Definición

La Transformada de Fourier Cuántica sobre $\mathbb{Z}_{N}$, con $N = 2^n$, está definida por:

$$\mathrm{QFT}|j\rangle = \frac{1}{\sqrt{N}} \sum_{k=0}^{N-1} e^{2\pi i jk/N} |k\rangle$$

Su implementación eficiente sólo requiere $O(n^2)$ puertas (frente a $O(N \log N)$ de la FFT clásica), con el circuito:

$$\mathrm{QFT}_n = (H \otimes I^{\otimes n-1}) \cdot R_2 \cdot R_3 \cdots R_n \cdots (H)_{\text{qubit } n} \cdot \mathrm{SWAP}$$

donde $R_k$ es la puerta de fase controlada con fase $2\pi/2^k$.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector
from qiskit_aer import AerSimulator

from src.quantum_math import QuantumMath
from src.visualization import QuantumVisualization

print('Módulos cargados.')

## 4.2 Circuito de la QFT

In [ ]:
def qft_circuit(n: int, inverse: bool = False, swap: bool = True) -> QuantumCircuit:
    """Construye el circuito de la Transformada de Fourier Cuántica.

    Parámetros
    ----------
    n : int
        Número de qubits.
    inverse : bool
        Si True, construye la QFT inversa.
    swap : bool
        Si True, añade puertas SWAP finales para invertir el orden de los qubits.

    Retorna
    -------
    QuantumCircuit
    """
    qc = QuantumCircuit(n, name='QFT' if not inverse else 'QFT†')

    def _qft_recursive(qc: QuantumCircuit, k: int):
        if k < 0:
            return
        qc.h(k)
        for j in range(k - 1, -1, -1):
            phase = 2 * np.pi / (2 ** (k - j + 1))
            qc.cp(phase, j, k)
        _qft_recursive(qc, k - 1)

    _qft_recursive(qc, n - 1)

    if swap:
        for i in range(n // 2):
            qc.swap(i, n - i - 1)

    if inverse:
        qc = qc.inverse()
        qc.name = 'QFT†'

    return qc

# Circuito para n=4
n = 4
qc_qft = qft_circuit(n)
print(f'Circuito QFT para {n} qubits:')
print(qc_qft.draw('text'))
print(f'\nTotal de operaciones: {qc_qft.size()}')

## 4.3 Verificación: comparación con la matriz analítica

In [ ]:
# Matriz del circuito Qiskit
n = 3
qc_qft_3 = qft_circuit(n)
U_qiskit = Operator(qc_qft_3).data

# Matriz analítica
U_analytic = QuantumMath.qft_matrix(n)

# Diferencia
diff = np.max(np.abs(U_qiskit - U_analytic))
print(f'Diferencia máxima |U_Qiskit - U_analítica| = {diff:.2e}')
print(f'¿Son iguales (tol=1e-10)? {np.allclose(U_qiskit, U_analytic, atol=1e-10)}')

# Visualización de la matriz
fig = QuantumVisualization.plot_unitary(
    U_analytic, title=f'QFT para {n} qubits'
)
plt.show()

## 4.4 Ejemplo numérico: QFT sobre un estado de prueba

In [ ]:
# Estado de entrada: |5〉 = |101〉 para n=3
n = 3
j_input = 5

# Preparar el estado |j〉 como vector de longitud 2^n
state_in = np.zeros(2**n, dtype=complex)
state_in[j_input] = 1.0
print(f'Estado de entrada: |{j_input}〉 = |{format(j_input, f"0{n}b")}〉')

# Aplicar QFT analítica
U_qft = QuantumMath.qft_matrix(n)
state_out = U_qft @ state_in

print(f'\nEstado de salida QFT|{j_input}〉:')
N = 2**n
for k, amp in enumerate(state_out):
    phase_exact = 2 * np.pi * j_input * k / N
    print(f'  |{format(k, f"0{n}b")}〉: {amp:.4f} '
          f'(fase = e^{{i·{phase_exact:.2f}}} = {np.exp(1j*phase_exact):.4f})')

# Visualización
fig = QuantumVisualization.plot_state_vector(state_out, title=f'QFT|{j_input}〉')
plt.show()

In [ ]:
# Verificar QFT · QFT† = I
n = 4
qc_qft_n   = qft_circuit(n, inverse=False)
qc_iqft_n  = qft_circuit(n, inverse=True)

U_fwd = Operator(qc_qft_n).data
U_inv = Operator(qc_iqft_n).data

product = U_fwd @ U_inv
is_identity = np.allclose(product, np.eye(2**n), atol=1e-10)
print(f'QFT · QFT† = I para n={n}: {is_identity}')

## 4.5 Escalabilidad: número de puertas vs n

In [ ]:
n_list   = list(range(2, 14))
gates_qft = [qft_circuit(n).size() for n in n_list]
gates_th  = [n * (n + 1) // 2 + n // 2 for n in n_list]  # O(n²) teórico

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(n_list, gates_qft, 'o-', color='#58a6ff', linewidth=2, label='Qiskit (medido)')
ax.plot(n_list, gates_th,  's--', color='#f78166', linewidth=1.5, label='O(n²) teórico')
ax.set_xlabel('Número de qubits (n)')
ax.set_ylabel('Número de puertas')
ax.set_title('Escalabilidad del circuito QFT')
ax.legend()
ax.grid(alpha=0.3)
ax.set_facecolor('#161b22')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 4.6 Ejercicios propuestos

1. Aplica la QFT al estado $|+\rangle^{\otimes 3}$ y determina el estado resultante. ¿Cuál es su interpretación geométrica?

2. Implementa la QFT inversa sin usar `.inverse()` de Qiskit, invirtiendo manualmente el orden de las puertas y negando los ángulos de las rotaciones controladas.

3. Demuestra la propiedad de **periodicidad**: si el estado de entrada es $\frac{1}{\sqrt{r}} \sum_{j=0}^{r-1} |j \cdot N/r\rangle$, la QFT produce otro estado periódico. Ilustra con $N=8$, $r=2$.

4. Investiga qué ocurre con la QFT cuando se limita la precisión de las fases $R_k$ a $k \leq k_{\max}$. ¿Cuántos pasos son necesarios para mantener fidelidad $> 0.99$?